In [2]:
import torch
import torch.nn as nn
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os, requests
from io import BytesIO
from typing import List

def download_sample_images(save_dir:str) -> List:
    '''샘플 이미지 다운로드'''
    sample_images = {
        'cat.jpg': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg',
        'dog.jpg': 'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg',
        'bird.jpg': 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/45/Eopsaltria_australis_-_Mogo_Campground.jpg/1200px-Eopsaltria_australis_-_Mogo_Campground.jpg',
    }
    downloaded_images = []
    os.makedirs(save_dir, exist_ok=True)
    for filename, url in sample_images.items() :
        filepath = os.path.join(save_dir, filename)
        if os.path.exists(filepath):
            print(f"이미 존재 : {filepath}")
            downloaded_images.append(filepath)
            continue
        try:
            print(f"다운로드 : {filename}")
            response = requests.get(url)
            response.raise_for_status()
            Image.open(BytesIO(response.content))
            image = Image.convert('RGB')
            image.save(filepath)
            print(f'저장완료 : {filepath}')
            downloaded_images.append(filepath)
        except Exception as e:
            print(f"다운로드 실패 : {e}")

    return downloaded_images

download_sample_images('./14vit_image')



# 기본이미지 로딩
def basic_image_loading(image_path:str):
    '''기본이미지 로딩 방법'''
    img = Image.open(image_path)
    print(f"===이미지 정보====")
    print(f"이미지 모드 : {img.mode}")
    print(f"이미지 크기 : {img.size}")   # (W,H) # PIL Image에는 .shape가 없음


    # numpy 배열로 반환
    img_array = np.array(img)
    print(f"===Numpy 배열 반환====")
    print(f"배열 사이즈 : {img_array.shape}")  # (H, W, Channel)
    print(f"데이터 타입 : {img_array.dtype}")
    print(f"값 범위 : {img_array.min()} ~ {img_array.max()}")

    return img


def vi_standard_preprocessing(img):
    '''vit 표준 전처리 파이프라인'''
    image_size = 244
    mean = [0.5, 0.5, 0.5]  # ImageNet의 평균
    std = [0.2, 0.2, 0.2]  # ImageNet 표준편차

    # 전처리 파이프라인
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])
    print(f"\n=====전처리 결과=====")
    print(f"원본이미지 크기 : {img.size}")
    img_tensor = preprocess(img)
    print(f"전처리 후 크기 : {img_tensor.shape}")
    print(f"전처리 후 값 범위 : {img_tensor.min()} ~ {img_tensor.max()}")

    # 배치차원 추가
    img_batch = img_tensor.unsqueeze(0)
    print(f"배치처리 후 크기 : {img_batch.shape}")

    return img_tensor, preprocess


# 학습용 데이터 증강
def training_augmentation(img):
    """학습 시 사용하는 데이터 증강"""    
    IMAGE_SIZE = 224
    MEAN = [0.485, 0.456, 0.406]
    STD = [0.229, 0.224, 0.225]
    
    # 학습용 증강 파이프라인
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),  # 랜덤 크롭
        transforms.RandomHorizontalFlip(p=0.5),                      # 좌우 반전
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # 색상 변형
        transforms.RandomRotation(degrees=15),                       # 회전
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ])
    
    # 평가용 파이프라인 (증강 없음)
    val_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ])
    
    print(f"\n[학습용 증강 기법]")
    print(f"  1. RandomResizedCrop: 랜덤 위치/크기로 자르기")
    print(f"  2. RandomHorizontalFlip: 50% 확률로 좌우 반전")
    print(f"  3. ColorJitter: 밝기, 대비, 채도 변형")
    print(f"  4. RandomRotation: +-15도 회전")
    
    # 같은 이미지에 여러 번 증강 적용
    print(f"\n[동일 이미지에 증강 적용 예시]")
    augmented_images = []
    for i in range(4):
        aug_img = train_transform(img)
        augmented_images.append(aug_img)
        print(f"  증강 {i+1}: shape={aug_img.shape}, "
              f"min={aug_img.min():.3f}, max={aug_img.max():.3f}")
    
    return train_transform, val_transform, augmented_images





if __name__ == '__main__' :
    sample_images = download_sample_images('./14vit_image/')
    for img in sample_images:
        img = basic_image_loading(img)
        img_tensor, preprocess = vi_standard_preprocessing(img)
        train_transform, val_transform, augmented_images = training_augmentation(img)

이미 존재 : ./14vit_image\cat.jpg
이미 존재 : ./14vit_image\dog.jpg
이미 존재 : ./14vit_image\bird.jpg
이미 존재 : ./14vit_image/cat.jpg
이미 존재 : ./14vit_image/dog.jpg
이미 존재 : ./14vit_image/bird.jpg
===이미지 정보====
이미지 모드 : RGB
이미지 크기 : (1200, 1198)
===Numpy 배열 반환====
배열 사이즈 : (1198, 1200, 3)
데이터 타입 : uint8
값 범위 : 0 ~ 255

=====전처리 결과=====
원본이미지 크기 : (1200, 1198)
전처리 후 크기 : torch.Size([3, 244, 244])
전처리 후 값 범위 : -2.5 ~ 2.5
배치처리 후 크기 : torch.Size([1, 3, 244, 244])

[학습용 증강 기법]
  1. RandomResizedCrop: 랜덤 위치/크기로 자르기
  2. RandomHorizontalFlip: 50% 확률로 좌우 반전
  3. ColorJitter: 밝기, 대비, 채도 변형
  4. RandomRotation: +-15도 회전

[동일 이미지에 증강 적용 예시]
  증강 1: shape=torch.Size([3, 224, 224]), min=-2.118, max=2.379
  증강 2: shape=torch.Size([3, 224, 224]), min=-2.118, max=2.239
  증강 3: shape=torch.Size([3, 224, 224]), min=-2.118, max=2.640
  증강 4: shape=torch.Size([3, 224, 224]), min=-2.118, max=1.943
===이미지 정보====
이미지 모드 : RGB
이미지 크기 : (1200, 989)
===Numpy 배열 반환====
배열 사이즈 : (989, 1200, 3)
데이터 타입 : uint8
값 범위 : 0 ~ 255

===